# Car Evaluation Classification

1,728 cars, each described by six categorical attributes — buying price, maintenance price, number of doors, persons capacity, luggage-boot size and safety. Predict the overall acceptability: unacceptable, acceptable, good or very good — a deliberately imbalanced, all-categorical problem.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Car Evaluation (id 19)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/19/data.csv')))
print(df.shape)
print('Cars per acceptability class:')
print(df['class'].value_counts())
df.head(3)


(1728, 7)
Cars per acceptability class:
class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc


In [2]:

# All six features are categorical, so one-hot encode them before splitting
# (categories are fixed labels, not target information).
X = pd.get_dummies(df.drop(columns=['class'])); y = df['class']
feats = list(X.columns)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9722
Logistic Regression accuracy: 0.9306
              precision    recall  f1-score   support

         acc       0.91      0.97      0.94        96
        good       1.00      0.94      0.97        17
       unacc       0.99      0.98      0.99       303
       vgood       1.00      0.88      0.93        16

    accuracy                           0.97       432
   macro avg       0.98      0.94      0.96       432
weighted avg       0.97      0.97      0.97       432



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Top 20 one-hot car attributes')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


safety_low        0.140428
persons_2         0.129441
safety_high       0.072645
persons_4         0.049759
buying_low        0.047979
maint_vhigh       0.047609
safety_med        0.047507
lug_boot_small    0.045546
buying_vhigh      0.045307
persons_more      0.043445
maint_low         0.040411
buying_high       0.040328
maint_high        0.038270
maint_med         0.036601
buying_med        0.036321
doors_2           0.033436
lug_boot_big      0.033212
lug_boot_med      0.020784
doors_5more       0.017926
doors_4           0.016894
dtype: float64


Winner: Random Forest
